# 🫀 실험 20c — **라벨을 고치고 처음부터 다시 채점한다** (학습 0회)

**MedKOS / `notebooks/exp20c_relabel_rescore.ipynb`** · 퀘스트 `ailab-2026-0015`

실험20b 가 **PTBDB 의 "anterior" 는 우리 `AMI` 가 아니라 `ASMI` 에 가깝다**를
사전등록 검정으로 보였다(P-4 +0.1934, 이중차분 +0.2736). 그러면 실험20·20b 의
**모든 숫자가 오염된 라벨 위에서 계산된 것**이다. 고치고 다시 잰다.

> **사후 스와핑은 하지 않는다.** "AMI 대신 ASMI 점수를 갖다 쓰니 0.865더라" 는
> 반칙이다. **crosswalk 를 문서로 먼저 정하고, 새 라벨 파일을 만들고, 처음부터
> 다시 채점**한다. 그리고 **결론이 매핑 선택에 의존하는지**를 함께 본다.

## 이번에 함께 고치는 오염 세 가지

| 무엇 | 왜 틀렸나 | 어떻게 고치나 |
|---|---|---|
| **라벨** | "anterior" 를 `AMI` 로 채점 | crosswalk v2 (좁은/넓은 **둘 다** 보고) |
| **OR 구성** | `GMIN_PT=20` 으로 채점 제외한 `IPLMI`·`ALMI`·`LMI` 를 **OR 에는 그대로 넣었다** | 채점 가능 부위만으로 OR 도 함께 낸다 |
| **부분적중 0.803** | 무작위 기준선이 없다. 환자당 3.33개를 켜면 우연히도 맞는다 | **순열 영가설**과 함께 보고 |
| **CI 스케일** | 특이도 CI 가 `[-0.183,+0.575]`, 오즈비가 `[-0.47,+3.63]` | 유계는 **logit**, 비율은 **log** |

## crosswalk v2 — 근거를 적고 셋을 **모두** 보관한다

`ailab-2026-0016` 에서 QRS 110 vs 120 ms 충돌을 임의로 고르지 않고 둘 다 남긴 것과
같은 처리다. 실험3 에서도 결론이 두 값 사이에 갇히는지 확인했다.

| 매핑 | "anterior" → | 근거 |
|---|---|---|
| **v1 (원래)** | `AMI` | 단어를 그대로 옮김. **사전등록된 것이라 기록에 남긴다** |
| **v2-narrow** | `ASMI` | PTB-XL SCP: `ASMI` = V1–V4 Q파 / `AMI` = V3–V4. 1990년대 PTBDB 자유서술의 "anterior" 는 전중격을 포함하는 넓은 뜻. 실험20b 가 측정으로 지지 |
| **v2-wide** | `{ASMI, AMI}` | 어느 쪽인지 단정하지 않고 둘 다 양성으로 둔다(다중라벨이라 가능) |

**결론이 v2-narrow 와 v2-wide 사이에 갇히면** 매핑 선택에 강건한 것이고,
갈리면 **갈린다고 보고**한다. 어느 쪽도 임의로 고르지 않는다.

## 사전등록

| 관문 | 내용 | 지지 조건 |
|---|---|---|
| **P-1★★** | 매핑을 고치면 전벽 계열 외부 AUROC 가 올라가나 | v2-narrow 의 전벽 AUROC − v1 `AMI` **> 0.10** |
| **P-2** | 결론이 매핑 선택에 강건한가 | v2-narrow 와 v2-wide 의 차 **< 0.05** |
| **P-3★★** | 채점 가능 부위만으로 OR 하면 rule-in 이 살아나나 | `LR+ ≥ 2.0` (오염된 7부위 OR 은 1.23) |
| **P-4★★** | 부분적중이 무작위보다 나은가 | 순열 영가설 대비 **> 0** (짝지은 부트스트랩 CI) |
| **P-5** | 알람률 고정 임계값이 시드 변동을 줄이나 | 임계값 변동계수 CV **< 0.20** (민감도 고정은 ~0.9) |

## 하지 않는 것

- 새 학습 · 새 구성 · 외부에서의 임계값 재조정
- v1 결과 삭제 (**사전등록된 것이므로 그대로 남긴다** — v2 는 교정본으로 병기)


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · "
      "assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정 (학습 0회 · GPU 불필요)
!pip -q install wfdb

import os, sys, json, time, re, ast, subprocess, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~22 와 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, NMIN = 5, 20, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
LEADS = {"I+II+V2+V5": [0, 1, 7, 10], "12": list(range(12))}
# ★★ 여기까지

DEPLOY, REF = "I+II+V2+V5", "12"
SEEDS = [0, 1, 2]
GMIN_PT, GMIN_SUB = 20, 20
SENS_TARGET, ALARM_RATE = 0.90, 0.20     # 알람률 고정 동작점(_t_for_rate 이식)
GAIN_THR, ROBUST_THR, LRP_THR, CV_THR = 0.10, 0.05, 2.0, 0.20
BOOT = 2000

# ── crosswalk. **셋을 모두 계산해서 결론이 선택에 의존하는지 본다.**
CROSSWALK = {
    "v1(원래·사전등록)": {"anterior": ["AMI"]},
    "v2-narrow":        {"anterior": ["ASMI"]},
    "v2-wide":          {"anterior": ["ASMI", "AMI"]},
}
CROSSWALK_RATIONALE = {
    "v1(원래·사전등록)": "단어를 그대로 옮긴 것. 사전등록된 매핑이므로 기록에 남긴다",
    "v2-narrow": ("PTB-XL SCP 는 ASMI=V1–V4 Q파 / AMI=V3–V4 로 가른다. PTBDB 는 1990년대 "
                  "임상 요약문의 자유서술이고 거기서 'anterior MI' 는 전중격을 포함하는 "
                  "넓은 뜻으로 쓰였다(anterior 와 anteroseptal 이 전통적으로 혼용). "
                  "실험20b P-4 +0.1934 · 이중차분 +0.2736 이 이를 측정으로 지지"),
    "v2-wide": "어느 쪽인지 단정하지 않고 둘 다 양성으로 둔다(다중라벨이라 가능)",
}

CONFIG = dict(exp="exp20c_relabel_rescore", quest="ailab-2026-0015",
              parent_exp=["exp20_ptbdb", "exp20b_patient"],
              purpose=("실험20b 가 라벨 정의 불일치를 확정했으므로 crosswalk 를 다시 쓰고 "
                       "**처음부터 다시 채점**한다. 사후 스와핑이 아니다"),
              dataset="PTB Diagnostic ECG Database (PhysioNet, Open Access)",
              change_one_thing="새 학습 없음. 라벨 crosswalk 와 채점 절차만 교정",
              deploy=DEPLOY, seeds=SEEDS, crosswalk=CROSSWALK,
              crosswalk_rationale=CROSSWALK_RATIONALE,
              gmin_patients=GMIN_PT, sens_target=SENS_TARGET, alarm_rate=ALARM_RATE,
              ci_rule="유계 지표는 logit · 비율/오즈비는 log 스케일에서 CI 를 잡고 되돌린다",
              predictions={
                  "P-1": f"v2-narrow 전벽 AUROC − v1 AMI > {GAIN_THR}",
                  "P-2": f"|v2-narrow − v2-wide| < {ROBUST_THR} (매핑 선택에 강건)",
                  "P-3": f"채점 가능 부위만 OR 하면 LR+ >= {LRP_THR} (오염판은 1.23)",
                  "P-4": "부분적중이 순열 영가설보다 높다 (짝지은 부트스트랩 CI > 0)",
                  "P-5": f"알람률 고정 임계값의 시드 변동계수 CV < {CV_THR}"},
              caveat=("v1 결과는 삭제하지 않는다 — 사전등록된 것이라 기록으로 남기고 "
                      "v2 를 교정본으로 병기한다"))
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp20c_relabel", CONFIG, project=PROJECT)

from scipy import stats
def t_ci(v, conf=.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2:
        return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

def t_ci_logit(v, conf=.95):
    """유계 지표(0~1)는 logit 에서 CI 를 잡는다 — 실험20b 특이도 CI 가 음수로 나왔다."""
    v = np.clip(np.asarray([x for x in v if np.isfinite(x)], float), 1e-6, 1 - 1e-6)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v / (1 - v)), conf)
    f = lambda x: float(1 / (1 + np.exp(-x)))
    return f(m), f(lo), f(hi)

def t_ci_log(v, conf=.95):
    """비율·오즈비는 log 에서 — 실험20b 오즈비 CI 가 [-0.47,+3.63] 로 나왔다."""
    v = np.asarray([x for x in v if np.isfinite(x) and x > 0], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v), conf)
    return float(np.exp(m)), float(np.exp(lo)), float(np.exp(hi))

def t_for_rate(s, rate):
    """예측양성률 = rate 가 되는 임계값(상위 rate 분위).

    ★ `mit-bih/colab_step69_ratepoint.py::_t_for_rate` 를 그대로 옮겼다.
      민감도를 고정하면 유병률 0.2% 부위에서 임계가 점수 분포의 **꼬리**(1e-4)로
      날아가 사실상 '전원 양성' 이 된다. 알람률을 고정하면 그 일이 안 생긴다.
      음성 앵커라 데이터셋이 바뀌어도 전이가 안정적이고, **라벨이 필요 없다.**
    """
    return float(np.quantile(s, 1.0 - rate))

# ── 부모 실행
REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if os.path.isdir(r.get("dir", "")):
        DIRS[r.get("exp_id")] = r["dir"]
if "exp20_ptbdb" not in DIRS:
    raise RuntimeError(f"실험20 실행을 못 찾았다. 발견: {sorted(DIRS)}")
D20 = DIRS["exp20_ptbdb"]
PARENTS = [DIRS[k] for k in ("exp19_two_stage", "exp18_confirm",
                             "exp17_wearable", "exp16_four_lead") if k in DIRS]
def arm_at(d, n):
    p = os.path.join(d, "arms", n, "probs.npy")
    return np.load(p) if os.path.exists(p) else None
def find_arm(n):
    for d in PARENTS:
        a = arm_at(d, n)
        if a is not None:
            return a
    return None
SITES18 = json.load(open(os.path.join(DIRS["exp18_confirm"], "result.json"),
                         encoding="utf-8"))["sites"]
run.log(f"실험20: {D20}\n부위 순서: {SITES18}")


In [ ]:
# CELL 2 — 라벨 세 벌 만들기 (v1 · v2-narrow · v2-wide) + 환자 단위 집계
import pandas as pd

LBL = run.data("ptbdb_labels_v1.json")     # 실험20b 가 만든 헤더 캐시(원문 포함)
if not os.path.exists(LBL):
    raise RuntimeError(f"라벨 캐시가 없다: {LBL} — 실험20b CELL 2 를 먼저 돌릴 것")
H = pd.DataFrame(json.load(open(LBL, encoding="utf-8")))
ACOL = next(c for c in H.columns if c.startswith("acute infarction"))
FCOL = next(c for c in H.columns if c.startswith("former infarction"))

def norm_loc(s):
    return re.sub(r"[^a-z]", "", str(s).strip().lower())

BASE_MAP = {"anteroseptal": ["ASMI"], "anteriorseptal": ["ASMI"],
            "anterolateral": ["ALMI"], "anteriorlateral": ["ALMI"],
            "anteroapicallateral": ["ALMI"], "anteroseptallateral": ["ASMI"],
            "anteroseptolateral": ["ASMI"], "inferior": ["IMI"],
            "inferolateral": ["ILMI"], "inferiorlateral": ["ILMI"],
            "inferoposterolateral": ["IPLMI"], "inferoposterlateral": ["IPLMI"],
            "inferiorposteriorlateral": ["IPLMI"], "lateral": ["LMI"],
            "inferolatera": ["ILMI"], "anterioranterior": ["AMI"],
            "inferoposteriorinferior": ["IMI"]}
LOC_DROP = {"no", "nein", "unknown", "", "none", "na", "inferoposterior",
            "inferiorposterior", "posterior", "posterolateral", "posteriorlateral"}

def build_map(cw):
    """crosswalk 를 적용한 전체 매핑. 'anteriorinferior' 도 함께 따라 움직인다."""
    m = dict(BASE_MAP)
    m["anterior"] = list(cw["anterior"])
    m["anteriorinferior"] = sorted(set(cw["anterior"]) | {"IMI"})   # 전벽+하벽 동시
    return m

SIG = run.data("ptbdb_12lead_100hz_v2.npz")
if not os.path.exists(SIG):
    SIG = run.data("ptbdb_12lead_100hz.npz")
RKEEP = [str(x) for x in np.load(SIG, allow_pickle=True)["recs"]]
HK = H.set_index("rec").loc[RKEEP]
PT = HK.patient.values
PTS = sorted(set(PT)); PIDX = {p: np.where(PT == p)[0] for p in PTS}

def labels_for(cw_name):
    m = build_map(CROSSWALK[cw_name])
    def sites_of(v):
        return sorted(set(m.get(norm_loc(v), [])))
    unk = sorted({norm_loc(v) for v in list(HK[ACOL].fillna("")) + list(HK[FCOL].fillna(""))
                  if norm_loc(v) not in m and norm_loc(v) not in LOC_DROP})
    if unk:
        raise LabelVocabError(f"[{cw_name}] 미매핑 {unk}")
    S = [sorted(set(sites_of(a)) | set(sites_of(f)))
         for a, f in zip(HK[ACOL].fillna(""), HK[FCOL].fillna(""))]
    YE = np.stack([[s in row for s in SITES18] for row in S]).astype(bool)
    Y_PT = np.stack([np.array([YE[PIDX[p], j].any() for p in PTS])
                     for j in range(len(SITES18))], axis=1)
    return YE, Y_PT

LAB = {k: labels_for(k) for k in CROSSWALK}
run.log("【라벨 세 벌】 환자 단위 양성 수 (레코드 549 · 환자 %d)" % len(PTS))
run.log(f"  {'부위':<8}" + "".join(f"{k:>22}" for k in CROSSWALK))
for j, s in enumerate(SITES18):
    run.log(f"  {s:<8}" + "".join(f"{int(LAB[k][1][:, j].sum()):>22}" for k in CROSSWALK))
IS_MI = {k: LAB[k][1].any(axis=1) for k in CROSSWALK}
for k in CROSSWALK:
    run.log(f"  {k:<20} MI 환자 {int(IS_MI[k].sum())}명 · "
            f"채점 가능 부위(>= {GMIN_PT}명): "
            f"{[s for j, s in enumerate(SITES18) if LAB[k][1][:, j].sum() >= GMIN_PT]}")

# ── 외부 arm (실험20 저장) · 환자 단위 점수
PE = {}
for c in LEADS:
    for sd in SEEDS:
        a = arm_at(D20, f"ext_{c}_s{sd}")
        if a is None:
            raise RuntimeError(f"실험20 arm ext_{c}_s{sd} 없음")
        assert_arm_shape(a, len(RKEEP), name=f"ext_{c}_s{sd}")
        PE[(c, sd)] = a
S_PT = {(c, sd): np.stack([np.array([PE[(c, sd)][PIDX[p], j].mean() for p in PTS])
                           for j in range(len(SITES18))], axis=1)
        for c in LEADS for sd in SEEDS}
run.log(f"\n외부 arm {len(PE)}개 · 환자 점수 {S_PT[(DEPLOY, SEEDS[0])].shape}")


In [ ]:
# CELL 3 — 【P-1·P-2】 매핑을 고치면 전벽이 살아나나, 그리고 선택에 강건한가
from sklearn.metrics import roc_auc_score

def auc_site(y, s):
    return float(roc_auc_score(y, s)) if y.any() and (~y).any() else np.nan

run.log("\n" + "=" * 112)
run.log("【P-1·P-2】 crosswalk 세 벌로 **처음부터 다시** 채점 (사후 스와핑 아님)")
run.log("=" * 112)
AUC = {}
for k in CROSSWALK:
    Y = LAB[k][1]
    AUC[k] = {}
    for j, s in enumerate(SITES18):
        AUC[k][s] = [auc_site(Y[:, j], S_PT[(DEPLOY, sd)][:, j]) for sd in SEEDS]
run.log(f"  {'부위':<8}" + "".join(f"{k:>22}" for k in CROSSWALK) + f"{'환자수(v2n)':>12}")
for j, s in enumerate(SITES18):
    n2 = int(LAB["v2-narrow"][1][:, j].sum())
    row = "".join(
        (f"{np.nanmean(AUC[k][s]):>22.4f}" if np.isfinite(np.nanmean(AUC[k][s]))
         else f"{'—':>22}") for k in CROSSWALK)
    run.log(f"  {s:<8}{row}{n2:>12}")

# 전벽 계열의 "대표 성능": 각 매핑에서 'anterior' 가 배정된 부위의 AUROC
def frontal_auc(k):
    tgt = CROSSWALK[k]["anterior"]
    per_seed = []
    for i in range(len(SEEDS)):
        vals = [AUC[k][s][i] for s in tgt if np.isfinite(AUC[k][s][i])]
        per_seed.append(float(np.mean(vals)) if vals else np.nan)
    return per_seed

FR = {k: frontal_auc(k) for k in CROSSWALK}
run.log("\n  'anterior' 가 배정된 부위의 AUROC (매핑별)")
for k in CROSSWALK:
    m, lo, hi = t_ci_logit(FR[k])
    run.log(f"    {k:<20} {m:.4f} [{lo:.4f},{hi:.4f}]  → {CROSSWALK[k]['anterior']}")
gain = [FR["v2-narrow"][i] - FR["v1(원래·사전등록)"][i] for i in range(len(SEEDS))]
mg, lg, hg = t_ci(gain)
robust = [abs(FR["v2-narrow"][i] - FR["v2-wide"][i]) for i in range(len(SEEDS))]
mr, lr_, hr = t_ci(robust)
run.log(f"\n  P-1 이득 (v2-narrow − v1) {mg:+.4f} [{lg:+.4f},{hg:+.4f}] vs 문턱 {GAIN_THR}")
run.log(f"  P-2 강건성 |v2-narrow − v2-wide| {mr:.4f} [{lr_:.4f},{hr:.4f}] vs 문턱 {ROBUST_THR}")
run.log("      ※ v2-wide 가 v2-narrow 와 크게 다르면 **결론이 매핑 선택에 의존**한다 —")
run.log("        그때는 어느 쪽도 고르지 않고 '갈린다' 고 보고한다")


In [ ]:
# CELL 4 — 【P-3】 채점 가능 부위만으로 OR — 오염 제거
#   ★ 실험20b 의 7부위 OR 은 `GMIN_PT=20` 으로 **이미 채점 제외한** IPLMI·ALMI·LMI 를
#     그대로 넣고 있었다. 임계값이 1e-4 꼬리로 날아간 헤드들이라 OR 전체를 끌어내린다.
#     사전등록된 제외 기준을 **일관되게** 적용한 판을 함께 낸다(새 기준이 아니다).
CW = "v2-narrow"
Y_PT = LAB[CW][1]; MI = IS_MI[CW]
SCORE_J = [j for j, s in enumerate(SITES18) if Y_PT[:, j].sum() >= GMIN_PT]
run.log("\n" + "=" * 112)
run.log(f"【P-3】 OR 구성 비교 ({CW} 라벨) — 채점 가능 부위 "
        f"{[SITES18[j] for j in SCORE_J]}")
run.log("=" * 112)

def thr_sens(score, pos, t=SENS_TARGET):
    p = score[pos]
    return float(np.quantile(p, 1.0 - t, method="lower")) if len(p) else -np.inf

# 내부(PTB-XL 5겹 OOF)에서 임계값을 동결한다 — 실험20b 와 같은 절차
PX = run.data("ptbxl_12lead_all.npz")
zx = np.load(PX, allow_pickle=True)
FOLD10, EID = zx["fold"], zx["eid"]
CV = (FOLD10 - 1) % K_FOLD
csv = "/content/ptbxl/ptbxl_database.csv"
if not os.path.exists(csv):
    os.makedirs("/content/ptbxl", exist_ok=True)
    subprocess.run(["wget", "-q", "-O", csv,
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"], check=True)
dfa = pd.read_csv(csv, index_col="ecg_id").loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
assert_label_vocab(SITES18, {c for cs in dfa.codes for c in cs}, kind="MI 부위 코드")
YI = np.stack([[s in c for s in SITES18] for c in dfa.codes]).astype(bool)

OOF = {}
for sd in SEEDS:
    o = np.zeros((len(EID), len(SITES18)), "float32")
    for j in range(len(SITES18)):
        for k in range(K_FOLD):
            a = find_arm(f"{DEPLOY}_s{sd}_f{k}")
            if a is None:
                raise RuntimeError(f"내부 OOF arm {DEPLOY}_s{sd}_f{k} 없음 · 뒤진 곳 {PARENTS}")
            o[np.where(CV == k)[0], j] = a[:, j]
    OOF[sd] = o
THR_S = {sd: np.array([thr_sens(OOF[sd][:, j], YI[:, j]) for j in range(len(SITES18))])
         for sd in SEEDS}
THR_R = {sd: np.array([t_for_rate(OOF[sd][:, j], ALARM_RATE) for j in range(len(SITES18))])
         for sd in SEEDS}

def or_metrics(cols, THR):
    out = []
    for sd in SEEDS:
        al = (S_PT[(DEPLOY, sd)][:, cols] >= THR[sd][cols]).any(axis=1)
        se = float(al[MI].mean()); sp = float((~al[~MI]).mean())
        out.append(dict(se=se, sp=sp, lrp=se / max(1 - sp, 1e-9),
                        lrn=(1 - se) / max(sp, 1e-9), rate=float(al.mean())))
    return out

run.log(f"  {'구성':<26}{'임계 방식':<12}{'Se':>8}{'Sp':>8}{'LR+':>8}{'LR−':>8}{'경보율':>9}")
VIEWS = {"7부위 OR (오염판)": list(range(len(SITES18))),
         "채점 가능 부위 OR": SCORE_J}
MET = {}
for vn, cols in VIEWS.items():
    for tn, TH in (("민감도 0.90", THR_S), (f"알람률 {ALARM_RATE}", THR_R)):
        r_ = or_metrics(cols, TH)
        MET[(vn, tn)] = r_
        run.log(f"  {vn:<26}{tn:<12}"
                f"{np.mean([x['se'] for x in r_]):>8.3f}"
                f"{np.mean([x['sp'] for x in r_]):>8.3f}"
                f"{np.exp(np.mean(np.log([x['lrp'] for x in r_]))):>8.2f}"
                f"{np.exp(np.mean(np.log([max(x['lrn'],1e-9) for x in r_]))):>8.3f}"
                f"{np.mean([x['rate'] for x in r_]):>9.1%}")
LRP = [x["lrp"] for x in MET[("채점 가능 부위 OR", "민감도 0.90")]]
m3, l3, h3 = t_ci_log(LRP)
run.log(f"\n  P-3 채점 가능 부위 OR 의 LR+ = {m3:.2f} [{l3:.2f},{h3:.2f}] "
        f"vs 문턱 {LRP_THR}  (오염판 실험20b 는 1.23)")


In [ ]:
# CELL 5 — 【P-4】 부분적중을 **무작위 기준선**과 함께 본다
#   0.803 은 환자당 3.33개를 켜고 얻은 값이다. 7개 중 3개를 켜면 우연히도 맞는다.
#   영가설: **경보 행을 환자 사이에서 셔플**한다(각 환자가 켠 개수는 보존).
run.log("\n" + "=" * 112)
run.log("【P-4】 부분적중 vs 순열 영가설")
run.log("=" * 112)
rs = np.random.RandomState(SEED0)

def hits(alarm, Y, mi):
    return float(((alarm & Y).any(axis=1))[mi].mean())

for vn, cols in VIEWS.items():
    obs, null = [], []
    for sd in SEEDS:
        al = S_PT[(DEPLOY, sd)][:, cols] >= THR_S[sd][cols]
        Yc = Y_PT[:, cols]
        obs.append(hits(al, Yc, MI))
        null.append(float(np.mean([hits(al[rs.permutation(len(PTS))], Yc, MI)
                                   for _ in range(200)])))
    mo, lo_, ho = t_ci(obs); mn, ln_, hn = t_ci(null)
    d = [obs[i] - null[i] for i in range(len(SEEDS))]
    md, ld, hd = t_ci(d)
    run.log(f"  {vn:<26} 실측 {mo:.3f} [{lo_:.3f},{ho:.3f}] · "
            f"무작위 {mn:.3f} · **차 {md:+.3f} [{ld:+.3f},{hd:+.3f}]**")
    if vn == "채점 가능 부위 OR":
        P4 = (md, ld, hd, mo, mn)
run.log("  ※ 셔플은 '각 환자가 몇 개를 켰나' 를 보존하므로 **경보 개수 효과를 상쇄**한다")
run.log("  ※ 완전일치는 다중라벨에서 지나치게 엄격해 지표로 쓰지 않는다 — Top-1 로 대체")
# Top-1: 점수 최고 부위가 참 라벨 집합에 들어가나
top1 = []
for sd in SEEDS:
    sc = S_PT[(DEPLOY, sd)][:, SCORE_J]
    rank = np.argsort(-sc, axis=1)[:, 0]
    top1.append(float(np.array([Y_PT[i, SCORE_J][rank[i]] for i in range(len(PTS))])[MI].mean()))
mt, lt, ht = t_ci_logit(top1)
run.log(f"  Top-1 정확도(채점 가능 부위) {mt:.3f} [{lt:.3f},{ht:.3f}] · "
        f"무작위 기준선 = 1/{len(SCORE_J)} 가 아니라 참 라벨 비율에 달림 → 아래 셔플로 비교")


In [ ]:
# CELL 6 — 【P-5】 알람률 고정 임계값이 시드 변동을 줄이나
run.log("\n" + "=" * 112)
run.log("【P-5】 임계값 결정 방식 — 민감도 고정 vs 알람률 고정")
run.log("=" * 112)
run.log("  민감도를 고정하면 유병률이 낮은 부위에서 임계가 점수 분포 **꼬리**로 날아간다.")
run.log("  (실험20b: IPLMI·LMI 임계 0.0000, 내부 특이도 0.480·0.539 = 사실상 전원 양성)")
run.log(f"\n  {'부위':<8}{'환자':>6}" + "".join(f"{'민감도 s'+str(s):>12}" for s in SEEDS)
        + f"{'CV':>7}" + "".join(f"{'알람률 s'+str(s):>12}" for s in SEEDS) + f"{'CV':>7}")
CVS, CVR = [], []
for j, s in enumerate(SITES18):
    a = np.array([THR_S[sd][j] for sd in SEEDS], float)
    b = np.array([THR_R[sd][j] for sd in SEEDS], float)
    cva = float(a.std(ddof=1) / a.mean()) if a.mean() > 0 else np.nan
    cvb = float(b.std(ddof=1) / b.mean()) if b.mean() > 0 else np.nan
    CVS.append(cva); CVR.append(cvb)
    run.log(f"  {s:<8}{int(Y_PT[:, j].sum()):>6}"
            + "".join(f"{x:>12.5f}" for x in a) + f"{cva:>7.2f}"
            + "".join(f"{x:>12.5f}" for x in b) + f"{cvb:>7.2f}")
mcs, mcr = float(np.nanmean(CVS)), float(np.nanmean(CVR))
run.log(f"\n  평균 CV — 민감도 고정 {mcs:.2f} · **알람률 고정 {mcr:.2f}** (문턱 {CV_THR})")
run.log("  낮을수록 '같은 코드·같은 데이터, 시드만 다름' 에서 동작점이 덜 흔들린다")


In [ ]:
# CELL 7 — 사전등록 채점
run.log("\n" + "=" * 112)
run.log("【사전등록 채점】")
run.log("=" * 112)
V = {}
V["P-1"] = decide(lg, hg, GAIN_THR, ">")
run.log(f"\n  P-1 v2-narrow − v1 > {GAIN_THR}")
run.log(f"      {mg:+.4f} [{lg:+.4f},{hg:+.4f}] → {MARK[V['P-1']]}")
run.log("      지지면 **실험20 의 전벽 숫자는 모델 성능이 아니라 라벨 오류였다**")

V["P-2"] = decide(lr_, hr, ROBUST_THR, "<")
run.log(f"\n  P-2 |v2-narrow − v2-wide| < {ROBUST_THR} (매핑 선택에 강건한가)")
run.log(f"      {mr:.4f} [{lr_:.4f},{hr:.4f}] → {MARK[V['P-2']]}")
run.log("      기각/미결이면 어느 쪽도 고르지 않고 **두 값을 함께** 보고한다")

V["P-3"] = decide(l3, h3, LRP_THR, ">")
run.log(f"\n  P-3 채점 가능 부위 OR 의 LR+ >= {LRP_THR}")
run.log(f"      {m3:.2f} [{l3:.2f},{h3:.2f}] → {MARK[V['P-3']]}  (오염판 1.23)")
run.log("      기각이면 **고장난 헤드 하나가 아니라 구조 문제**다 — OR 게이트 자체를 버려야 한다")

V["P-4"] = decide(P4[1], P4[2], 0.0, ">")
run.log(f"\n  P-4 부분적중 − 무작위 > 0")
run.log(f"      {P4[0]:+.3f} [{P4[1]:+.3f},{P4[2]:+.3f}] "
        f"(실측 {P4[3]:.3f} · 무작위 {P4[4]:.3f}) → {MARK[V['P-4']]}")

V["P-5"] = bool(mcr < CV_THR)
run.log(f"\n  P-5 알람률 고정 임계값의 평균 CV < {CV_THR}")
run.log(f"      {mcr:.2f} (민감도 고정 {mcs:.2f}) → {'✅ 지지' if V['P-5'] else '❌ 기각'}")

run.log("\n" + "=" * 112)
for k in ("P-1", "P-2", "P-3", "P-4", "P-5"):
    run.log(f"  {k}: {MARK[V[k]] if V[k] is not None else MARK[None]}")
run.log("  ⚠️ v1 결과는 **삭제하지 않는다** — 사전등록된 것이라 기록으로 남기고 v2 를 병기한다")
run.log("  ⚠️ 학습 0회. 새 숫자는 전부 같은 저장 arm 을 다시 채점한 것이다")


In [ ]:
# CELL 8 — 그림
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))

ks = list(CROSSWALK)
ax[0].bar(range(len(ks)), [np.nanmean(FR[k]) for k in ks],
          color=["#bbbbbb", "#2ca02c", "#1f77b4"])
ax[0].set_xticks(range(len(ks))); ax[0].set_xticklabels(ks, rotation=12, fontsize=8)
ax[0].axhline(.5, ls=":", c="k", lw=1); ax[0].set_ylim(0, 1)
ax[0].set_ylabel("'anterior' 배정 부위 AUROC")
ax[0].set_title(f"라벨 crosswalk · P-1 {MARK[V['P-1']]}")

lab = [f"{v}\n{t}" for v in VIEWS for t in ("민감도 0.90", f"알람률 {ALARM_RATE}")]
val = [np.exp(np.mean(np.log([x["lrp"] for x in MET[(v, t)]])))
       for v in VIEWS for t in ("민감도 0.90", f"알람률 {ALARM_RATE}")]
ax[1].bar(range(len(val)), val, color=["#bbbbbb", "#999999", "#2ca02c", "#1f77b4"])
ax[1].set_xticks(range(len(val))); ax[1].set_xticklabels(lab, fontsize=7)
ax[1].axhline(LRP_THR, ls=":", c="r", lw=1); ax[1].axhline(1.0, ls="-", c="k", lw=.8)
ax[1].set_ylabel("LR+"); ax[1].set_title(f"OR 구성 × 임계 방식 · P-3 {MARK[V['P-3']]}")

ax[2].bar(["민감도 고정", "알람률 고정"], [mcs, mcr], color=["#ff7f0e", "#2ca02c"])
ax[2].axhline(CV_THR, ls=":", c="r", lw=1)
ax[2].set_ylabel("임계값 시드 변동계수 CV")
ax[2].set_title(f"동작점 안정성 · P-5 {'✅' if V['P-5'] else '❌'}")
plt.tight_layout(); run.save_fig("exp20c_relabel_rescore", fig); plt.show()


In [ ]:
# CELL 9 — 결과 저장
res = {
    "week": 2, "exp_id": "exp20c_relabel", "quest": "ailab-2026-0015",
    "step": "exp20c-relabel-rescore", "split": "inter",
    "task": "라벨 crosswalk 재정의 후 전면 재채점 + 오염된 기준선 교정 (학습 0회)",
    "notebook": "notebooks/exp20c_relabel_rescore.ipynb",
    "metric": "frontal_auroc_v2narrow", "value": round(float(np.nanmean(FR["v2-narrow"])), 4),
    "passed": bool(V["P-1"] is True),
    "crosswalk": CROSSWALK, "crosswalk_rationale": CROSSWALK_RATIONALE,
    "frontal_auroc": {k: float(np.nanmean(FR[k])) for k in CROSSWALK},
    "gain_v2narrow_minus_v1": [float(x) for x in gain],
    "robustness_narrow_vs_wide": [float(x) for x in robust],
    "auroc_by_map": {k: {s: float(np.nanmean(AUC[k][s])) for s in SITES18} for k in CROSSWALK},
    "score_sites_v2narrow": [SITES18[j] for j in SCORE_J],
    "or_metrics": {f"{v}|{t}": [{kk: float(vv) for kk, vv in x.items()} for x in MET[(v, t)]]
                   for v in VIEWS for t in ("민감도 0.90", f"알람률 {ALARM_RATE}")},
    "hit_partial_vs_null": {"diff": float(P4[0]), "ci": [float(P4[1]), float(P4[2])],
                            "observed": float(P4[3]), "null": float(P4[4])},
    "top1": float(np.mean(top1)),
    "thr_cv_sensitivity": float(mcs), "thr_cv_alarmrate": float(mcr),
    "verdicts": {k: V[k] for k in ("P-1", "P-2", "P-3", "P-4", "P-5")},
    "caveats": [
        "학습 0회 — 실험20 의 저장 arm 을 다시 채점한 것",
        "v1(사전등록) 결과는 삭제하지 않고 병기한다",
        "채점 가능 부위 OR 은 새 기준이 아니라 GMIN_PT=20 을 일관되게 적용한 것",
        "알람률 고정은 mit-bih/colab_step69_ratepoint.py::_t_for_rate 이식",
        "본 모델은 급성 심근경색이 아니라 판독 라벨을 예측한다"],
}
run.save_json("result", res); run.finish(res)
print(json.dumps({k: res[k] for k in ("metric", "value", "verdicts", "frontal_auroc",
                                      "thr_cv_sensitivity", "thr_cv_alarmrate")},
                 ensure_ascii=False, indent=2))
